# Transformer temperatures with Temperature-Dependent Power Flow (TDPF)

The TDPF of pandapower solves the conductor temperature of the overhead lines, because the line
resistance depends on it. Transformers are treated differently: their winding resistance is not
corrected with the temperature, but the top-oil and hot-spot temperatures are interesting results on
their own, e.g. for loading and ageing assessment according to IEC 60076-7.

When `runpp` is called with `tdpf=True`, pandapower reports the top-oil and hot-spot temperature of
every in-service transformer in `net.res_trafo` and `net.res_trafo3w`. The calculation is based on the
[transformer-thermal-model](https://github.com/alliander-opensource/transformer-thermal-model) library,
which implements the IEC 60076-7 loading guide. It is an optional dependency:

```bash
pip install pandapower[tdpf]
```

If the package is not installed, the line TDPF still works unchanged; pandapower only logs a warning
and does not report transformer temperatures.

In [ ]:
import numpy as np
import pandas as pd

from pandapower import pandapowerNet, runpp
from pandapower.control import ConstControl
from pandapower.create import (
    create_buses,
    create_ext_grid,
    create_load,
    create_transformer3w_from_parameters,
    create_transformer_from_parameters,
)
from pandapower.timeseries import DFData, OutputWriter, run_timeseries

## A minimal example network

The thermal model only needs the transformer nameplate data that is already in the transformer table,
plus two optional columns:

| Column | Meaning | Default |
|---|---|---|
| `cooling_type` | cooling type of the transformer, `"ONAN"` or `"ONAF"` | `"ONAN"` |
| `air_temperature_degree_celsius` | ambient air temperature in °C | `35` |

The load losses, no-load losses and rated currents are derived from `sn_mva`, `vn_lv_kv` and
`vkr_percent` / `pfe_kw`.

In [ ]:
def create_trafo_net(cooling_type="ONAN", air_temperature_degree_celsius=35.0, p_mw=30.0):
    net = pandapowerNet(name="tdpf_trafo")
    hv_bus, lv_bus = create_buses(net, nr_buses=2, vn_kv=[110.0, 20.0])
    create_ext_grid(net, hv_bus)
    create_transformer_from_parameters(
        net,
        hv_bus=hv_bus,
        lv_bus=lv_bus,
        sn_mva=40.0,
        vn_hv_kv=110.0,
        vn_lv_kv=20.0,
        vk_percent=10.0,
        vkr_percent=0.5,
        pfe_kw=20.0,
        i0_percent=0.1,
        cooling_type=cooling_type,
        air_temperature_degree_celsius=air_temperature_degree_celsius,
    )
    create_load(net, lv_bus, p_mw=p_mw, q_mvar=0.2 * p_mw)
    return net


net = create_trafo_net()
runpp(net, tdpf=True)

net.res_trafo[
    [
        "loading_percent",
        "top_oil_temperature_degree_celsius",
        "hot_spot_temperature_degree_celsius",
        "temperature_degree_celsius",
    ]
]

`top_oil_temperature_degree_celsius` is the oil temperature at the top of the tank,
`hot_spot_temperature_degree_celsius` is the hottest point of the winding, and
`temperature_degree_celsius` is an alias of the hot-spot temperature so that the existing contingency
temperature limits can be reused. All three columns are only created when the thermal calculation
actually ran.

## Steady state and thermal inertia

Whether a steady state or a transient temperature is reported depends on the `tdpf_delay_s` parameter,
the same parameter that controls the thermal inertia of the lines:

* `tdpf_delay_s=None` (the default) is the steady state temperature for the current loading.
* A finite `tdpf_delay_s` is the temperature after this time has passed, starting from a cold
  transformer.

For a single power flow, the transient result at a short delay is much lower than the steady state,
because a transformer needs hours to warm up.

In [ ]:
delays_min = [5, 15, 30, 60, 120, 240, 480, None]
temperatures = []
for delay_min in delays_min:
    net = create_trafo_net()
    runpp(net, tdpf=True, tdpf_delay_s=None if delay_min is None else delay_min * 60)
    temperatures.append(net.res_trafo.at[0, "top_oil_temperature_degree_celsius"])

pd.Series(temperatures, index=[str(d) for d in delays_min], name="top oil (°C)")

The temperature approaches the steady state value asymptotically. The oil time constant of a
transformer is in the range of two to three hours, which is much slower than the thermal inertia of an
overhead line.

## Cooling type ONAN and ONAF

`ONAN` (natural oil, natural air) and `ONAF` (natural oil, forced air) share the same steady state
temperature, because the IEC loading guide uses the same top-oil rise for both. They differ in the time
constants: an `ONAF` transformer has shorter oil and winding time constants, so it heats up *faster*
and reaches its final temperature sooner.

In [ ]:
curve = {}
for cooling_type in ("ONAN", "ONAF"):
    values = []
    for delay_min in delays_min[:-1]:
        net = create_trafo_net(cooling_type=cooling_type)
        runpp(net, tdpf=True, tdpf_delay_s=delay_min * 60)
        values.append(net.res_trafo.at[0, "top_oil_temperature_degree_celsius"])
    curve[cooling_type] = values

curve = pd.DataFrame(curve, index=delays_min[:-1])
curve.index.name = "delay (min)"

ax = curve.plot(marker="o", figsize=(6, 3.5))
ax.set_ylabel("Top-oil temperature (°C)")
ax.set_title("ONAF warms up faster, same steady state");

In [ ]:
net = create_trafo_net(cooling_type="ONAF")
runpp(net, tdpf=True)
curve.loc["steady"] = net.res_trafo.at[0, "top_oil_temperature_degree_celsius"]
curve.round(2)

## Ambient temperature

The hot-spot temperature is a temperature *rise* on top of the ambient temperature, so a hotter day
shifts the whole curve upwards. The rated temperature rise is defined for an ambient temperature of
20 °C, while pandapower defaults to a conservative 35 °C.

In [ ]:
values = []
for ambient in (0.0, 20.0, 35.0, 45.0):
    net = create_trafo_net(air_temperature_degree_celsius=ambient)
    runpp(net, tdpf=True)
    values.append(net.res_trafo.at[0, "top_oil_temperature_degree_celsius"])

pd.Series(values, index=[0.0, 20.0, 35.0, 45.0], name="top oil (°C)")

## Three winding transformers

A three winding transformer has one top-oil temperature but three hot-spot temperatures, one per
winding. They are reported as `hot_spot_temperature_degree_celsius_hv`, `..._mv` and `..._lv`, while
`temperature_degree_celsius` is the maximum of the three, mirroring how `loading_percent` aggregates
the windings.

The `vkr_*_percent` columns of `net.trafo3w` are pairwise values relative to the minimum of the two
rated powers involved, which is exactly what the thermal model expects.

In [ ]:
net3w = pandapowerNet(name="tdpf_trafo3w")
hv_bus, mv_bus, lv_bus = create_buses(net3w, nr_buses=3, vn_kv=[110.0, 20.0, 10.0])
create_ext_grid(net3w, hv_bus)
create_transformer3w_from_parameters(
    net3w,
    hv_bus=hv_bus,
    mv_bus=mv_bus,
    lv_bus=lv_bus,
    sn_hv_mva=100.0,
    sn_mv_mva=50.0,
    sn_lv_mva=25.0,
    vn_hv_kv=110.0,
    vn_mv_kv=20.0,
    vn_lv_kv=10.0,
    vk_hv_percent=10.0,
    vk_mv_percent=8.0,
    vk_lv_percent=12.0,
    vkr_hv_percent=0.5,
    vkr_mv_percent=0.4,
    vkr_lv_percent=0.6,
    pfe_kw=20.0,
    i0_percent=0.1,
)
create_load(net3w, mv_bus, p_mw=40.0, q_mvar=8.0)
create_load(net3w, lv_bus, p_mw=20.0, q_mvar=4.0)

runpp(net3w, tdpf=True)

net3w.res_trafo3w[
    [
        "loading_percent",
        "top_oil_temperature_degree_celsius",
        "hot_spot_temperature_degree_celsius_hv",
        "hot_spot_temperature_degree_celsius_mv",
        "hot_spot_temperature_degree_celsius_lv",
        "temperature_degree_celsius",
    ]
]

## Time series: warming up and cooling down

In a single power flow, a finite `tdpf_delay_s` always starts from a cold transformer. In a time series
simulation, pandapower chains the top-oil temperature across the time steps: the result of the previous
time step is the initial condition of the next one. This is what makes the thermal inertia physically
meaningful.

In the following example the load ramps up linearly while the time step is 10 minutes.

In [ ]:
net = create_trafo_net()
n_steps = 48
load_profile = pd.DataFrame(
    {
        "p_mw": np.linspace(10.0, 40.0, n_steps),
        "q_mvar": np.linspace(2.0, 8.0, n_steps),
    }
)
ds = DFData(load_profile)
ConstControl(net, "load", "p_mw", net.load.index, "p_mw", ds, recycle=False)
ConstControl(net, "load", "q_mvar", net.load.index, "q_mvar", ds, recycle=False)

ow = OutputWriter(net, output_path=None, log_variables=[])
ow.log_variable("res_trafo", "top_oil_temperature_degree_celsius")
ow.log_variable("res_trafo", "temperature_degree_celsius")

run_timeseries(net, time_steps=range(n_steps), tdpf=True, tdpf_delay_s=10 * 60)

In [ ]:
top_oil = ow.output["res_trafo.top_oil_temperature_degree_celsius"][0]
hot_spot = ow.output["res_trafo.temperature_degree_celsius"][0]

ax = top_oil.plot(figsize=(6, 3.5), label="top oil")
hot_spot.plot(ax=ax, label="hot spot")
ax.set_xlabel("Time step (10 min)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Transformer warming up with a ramping load")
ax.legend();

Because the load ramps up continuously and the transformer is cold at the beginning, the temperatures
lag behind the instantaneous steady state value. The gap is the thermal inertia of the transformer, and
it is much larger than for a line: tens of minutes are not enough to reach the steady state.

The temperatures are reported, not fed back into the power flow. In contrast to the line TDPF, the
transformer resistance `r_sc` is never modified, so voltages, flows and losses of the network are the
same as with `tdpf=False`. Only the additional result columns differ.